# TMJ Binary Position Classifier — Detector-Based Crops

Ноутбук для обучения **бинарного классификатора** положения ВНЧС (центральное / нецентральное).

**Approach A**: NIfTI-кропы 128³ вокруг предсказаний детектора (вместо центральных кропов).  
**Approach B**: `BinaryFocalLoss` + калибровка порога через Youden's J.

Поддерживаемые среды: **Yandex DataSphere** (основная) · **Google Colab** · **Local**

In [ ]:
%pip install -q --upgrade scipy tqdm nibabel scikit-learn pydicom pylibjpeg pylibjpeg-libjpeg

import subprocess, sys
from pathlib import Path

FS = Path("/home/jupyter/filestore")  # filestore — единственное writable хранилище

LEFT_DETECTOR_URL  = "https://github.com/tzopiz/MasterProject/releases/download/heatmap-detector-v1/left_detector.pth"
RIGHT_DETECTOR_URL = "https://github.com/tzopiz/MasterProject/releases/download/heatmap-detector-v1/right_detector.pth"
LABELS_URL         = "https://github.com/tzopiz/MasterProject/releases/download/crops-v1/tmj_position_labels.json"
MANIFEST_URL       = "https://github.com/tzopiz/MasterProject/releases/download/crops-v1/manifest.json"

def wget(url, dest):
    dest = Path(dest)
    if dest.exists():
        print(f"  уже есть: {dest} ({dest.stat().st_size/1e6:.1f} MB)")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  скачиваю {dest.name}...")
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(dest), url], check=True)
    print(f"  готово: {dest.stat().st_size/1e6:.1f} MB")

if Path("/home/jupyter").exists():  # DataSphere
    wget(LEFT_DETECTOR_URL,  FS / "models" / "left_detector.pth")
    wget(RIGHT_DETECTOR_URL, FS / "models" / "right_detector.pth")
    wget(LABELS_URL,         FS / "tmj_position_labels.json")
    wget(MANIFEST_URL,       FS / "manifest.json")

In [ ]:
import os, sys
from pathlib import Path

IN_DATASPHERE = Path("/home/jupyter").exists()
IN_COLAB = "google.colab" in sys.modules
env_name = "DataSphere" if IN_DATASPHERE else ("Colab" if IN_COLAB else "Local")
print(f"Среда: {env_name}")

if IN_DATASPHERE:
    FS                  = Path("/home/jupyter/filestore")
    MANIFEST_PATH       = FS / "manifest.json"
    LABELS_PATH         = FS / "tmj_position_labels.json"
    LEFT_DETECTOR_PATH  = FS / "models" / "left_detector.pth"
    RIGHT_DETECTOR_PATH = FS / "models" / "right_detector.pth"
    CROPS_DIR           = FS / "detector_crops_v2"
    OUTPUT_DIR          = FS / "experiments"
    DATASET_ROOT        = None  # не используется: кропы скачиваются готовые

elif IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DATA_ROOT           = Path("/content/drive/MyDrive/tmj_data")
    MANIFEST_PATH       = DATA_ROOT / "dataset_public" / "manifest_private.json"
    LABELS_PATH         = DATA_ROOT / "tmj_position_labels.json"
    LEFT_DETECTOR_PATH  = DATA_ROOT / "models" / "left_detector.pth"
    RIGHT_DETECTOR_PATH = DATA_ROOT / "models" / "right_detector.pth"
    CROPS_DIR           = Path("/content/detector_crops_v2")
    OUTPUT_DIR          = Path("/content/experiments")
    DATASET_ROOT        = DATA_ROOT / "dataset_public"

else:  # Local
    ROOT                = Path(".").resolve().parent
    MANIFEST_PATH       = ROOT / "data" / "dataset_cbct_public" / "manifest_private.json"
    LABELS_PATH         = ROOT / "data" / "tmj_position_labels.json"
    LEFT_DETECTOR_PATH  = ROOT / "models" / "checkpoints" / "left_detector.pth"
    RIGHT_DETECTOR_PATH = ROOT / "models" / "checkpoints" / "right_detector.pth"
    CROPS_DIR           = ROOT / "data" / "detector_crops_v2"
    OUTPUT_DIR          = ROOT / "experiments"
    DATASET_ROOT        = ROOT / "data" / "dataset_cbct_public"

CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MANIFEST_PATH       : {MANIFEST_PATH}  exists={MANIFEST_PATH.exists()}")
print(f"LABELS_PATH         : {LABELS_PATH}  exists={LABELS_PATH.exists()}")
print(f"LEFT_DETECTOR_PATH  : {LEFT_DETECTOR_PATH}  exists={LEFT_DETECTOR_PATH.exists()}")
print(f"RIGHT_DETECTOR_PATH : {RIGHT_DETECTOR_PATH}  exists={RIGHT_DETECTOR_PATH.exists()}")
print(f"CROPS_DIR           : {CROPS_DIR}")

In [3]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda"); print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps"); print("MPS")
else:
    device = torch.device("cpu"); print("CPU")
print(f"PyTorch: {torch.__version__}")

GPU: Tesla V100-PCIE-32GB
PyTorch: 2.0.1+cu118


## 2. Label Table

In [ ]:
import json, random, logging
from pathlib import Path
from typing import Dict, List, Tuple

logger = logging.getLogger("tmj")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")


def map_sagittal(code):
    if code not in (1, 2, 3): raise ValueError(f"Invalid sagittal code: {code}")
    return code - 1


def map_frontal(code):
    if code not in (4, 5, 6): raise ValueError(f"Invalid frontal code: {code}")
    return code - 4


def build_index(manifest_path, labels_path, dataset_root, cache_path=None):
    with open(manifest_path, "r", encoding="utf-8") as f: manifest = json.load(f)
    with open(labels_path, "r", encoding="utf-8") as f: labels_data = json.load(f)
    label_by_name = {p["name_raw"].strip(): p["labels"] for p in labels_data["patients"]}
    records, skipped = [], 0
    for study in manifest["studies"]:
        name = study["patient_name"].strip()
        if name not in label_by_name: skipped += 1; continue
        lbl = label_by_name[name]
        records.append({
            "study_id": study["study_id"],
            "dicom_dir": str(Path(dataset_root) / study["study_id"]),
            "patient_name": name,
            "sag_right": map_sagittal(lbl["sagittal"]["right"]),
            "sag_left":  map_sagittal(lbl["sagittal"]["left"]),
            "fr_right":  map_frontal(lbl["frontal"]["right"]),
            "fr_left":   map_frontal(lbl["frontal"]["left"]),
        })
    logger.info("build_index: %d matched, %d skipped", len(records), skipped)
    return records


def binarize_labels(records, crop_dir):
    crop_dir = Path(crop_dir).resolve()
    out = []
    for rec in records:
        for side in ("left", "right"):
            out.append({
                "study_id": rec["study_id"],
                "patient_name": rec["patient_name"],
                "side": side,
                "sag": 0 if rec[f"sag_{side}"] == 0 else 1,
                "fr":  0 if rec[f"fr_{side}"]  == 0 else 1,
                "crop_path": str(crop_dir / rec["study_id"] / f"{rec['study_id']}_{side}.nii.gz"),
            })
    logger.info("binarize_labels: %d → %d binary records", len(records), len(out))
    return out


def split_by_patient(records, split_ratio=0.8, seed=42):
    patients = sorted(set(r["patient_name"] for r in records))
    rng = random.Random(seed); rng.shuffle(patients)
    n = len(patients)
    idx = min(max(1, int(n * split_ratio)), n - 1)
    train_p = set(patients[:idx]); val_p = set(patients[idx:])
    train = [r for r in records if r["patient_name"] in train_p]
    val   = [r for r in records if r["patient_name"] in val_p]
    logger.info("split: train=%d val=%d", len(train), len(val))
    return train, val


def split_by_patient_stratified(records, split_ratio=0.8, seed=42):
    """
    Split by patient ensuring both sag classes appear in val.
    Patients sorted by sag-positive fraction, then alternately assigned to train/val.
    """
    from collections import defaultdict
    patient_records = defaultdict(list)
    for r in records:
        patient_records[r["patient_name"]].append(r)

    # Compute sag=1 fraction per patient
    patients_info = []
    for name, recs in patient_records.items():
        frac = sum(r["sag"] for r in recs) / len(recs)
        patients_info.append((name, frac))

    # Sort by fraction, then interleave into train/val
    patients_info.sort(key=lambda x: x[1])
    rng = random.Random(seed)

    # Assign every Nth patient to val to spread class distribution
    n_val = max(1, int(len(patients_info) * (1 - split_ratio)))
    # Take evenly spaced indices for val
    step = len(patients_info) / n_val
    val_idx = set(int(i * step) for i in range(n_val))

    train_names = {p[0] for i, p in enumerate(patients_info) if i not in val_idx}
    val_names   = {p[0] for i, p in enumerate(patients_info) if i in val_idx}

    train = [r for r in records if r["patient_name"] in train_names]
    val   = [r for r in records if r["patient_name"] in val_names]

    # Log class distribution in each split
    for split_name, split_recs in [("Train", train), ("Val", val)]:
        n0 = sum(1 for r in split_recs if r["sag"] == 0)
        n1 = sum(1 for r in split_recs if r["sag"] == 1)
        logger.info("%s sag: %d central (%.0f%%) / %d non-central (%.0f%%)",
                    split_name, n0, 100*n0/max(len(split_recs),1),
                    n1, 100*n1/max(len(split_recs),1))
    return train, val


# DATASET_ROOT=None в DataSphere (DICOM не нужен — кропы уже готовы)
_dset_root = str(DATASET_ROOT) if DATASET_ROOT else str(CROPS_DIR)
all_records = build_index(str(MANIFEST_PATH), str(LABELS_PATH), _dset_root)
binary_records = binarize_labels(all_records, crop_dir=str(CROPS_DIR))

from collections import Counter
sag_dist = Counter(r["sag"] for r in binary_records)
fr_dist  = Counter(r["fr"]  for r in binary_records)
print(f"Записей: {len(all_records)} исследований → {len(binary_records)} кропов")
print(f"Sagittal: central={sag_dist[0]} non-central={sag_dist[1]}")
print(f"Frontal:  central={fr_dist[0]}  non-central={fr_dist[1]}")

In [ ]:
SPLIT_RATIO = 0.8
train_records, val_records = split_by_patient_stratified(binary_records, split_ratio=SPLIT_RATIO, seed=42)
print(f"Train: {len(train_records)}  Val: {len(val_records)}")

## 3. Preprocessing: Detector → Crops (один раз)

In [ ]:
existing = list(CROPS_DIR.rglob("*.nii.gz"))
print(f"Кропов уже есть: {len(existing)}, ожидается: {len(all_records)*2}")

if len(existing) < len(all_records) * 2:
    if IN_DATASPHERE:
        # DataSphere: скачиваем готовые кропы (сгенерированы heatmap-детектором v1)
        import subprocess, tarfile
        CROPS_URL = "https://github.com/tzopiz/MasterProject/releases/download/crops-v2/detector_crops_v2.tar.gz"
        tar_path = CROPS_DIR.parent / "detector_crops_v2.tar.gz"
        print("Скачиваю кропы crops-v2 (~800 MB)...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(tar_path), CROPS_URL], check=True)
        print("Распаковываю...")
        with tarfile.open(tar_path) as tf:
            tf.extractall(CROPS_DIR.parent)
        tar_path.unlink()
        n_crops = len(list(CROPS_DIR.rglob("*.nii.gz")))
        print(f"Готово: {n_crops} кропов")
    else:
        # Локально / Colab: генерируем из DICOM с heatmap-детекторами
        import torch.nn as nn
        import torch.nn.functional as F
        from scipy import ndimage
        import nibabel as nib
        from tqdm.notebook import tqdm

        TARGET_SHAPE = (96, 128, 128)

        # ── Inline model definition ──
        def _dconv(a, b):
            return nn.Sequential(
                nn.Conv3d(a,b,3,padding=1,bias=False), nn.BatchNorm3d(b), nn.ReLU(inplace=True),
                nn.Conv3d(b,b,3,padding=1,bias=False), nn.BatchNorm3d(b), nn.ReLU(inplace=True),
            )
        class _Enc(nn.Module):
            def __init__(self,a,b): super().__init__(); self.c=_dconv(a,b); self.p=nn.MaxPool3d(2)
            def forward(self,x): s=self.c(x); return self.p(s),s
        class _Dec(nn.Module):
            def __init__(self,a,b,c):
                super().__init__()
                self.u=nn.ConvTranspose3d(a,a//2,2,stride=2); self.c=_dconv(a//2+b,c)
            def forward(self,x,s):
                x=self.u(x)
                if x.shape!=s.shape: x=F.pad(x,[0,s.shape[4]-x.shape[4],0,s.shape[3]-x.shape[3],0,s.shape[2]-x.shape[2]])
                return self.c(torch.cat([s,x],1))
        class TMJHeatmapDetector(nn.Module):
            def __init__(self,feats=None):
                super().__init__()
                feats=feats or [32,64,128,256]
                self.encs=nn.ModuleList(); prev=1
                for f in feats: self.encs.append(_Enc(prev,f)); prev=f
                self.bot=_dconv(feats[-1],feats[-1]*2); prev=feats[-1]*2
                self.decs=nn.ModuleList()
                for f in reversed(feats): self.decs.append(_Dec(prev,f,f)); prev=f
                self.head=nn.Conv3d(feats[0],1,1)
            def forward(self,x):
                skips=[]
                for e in self.encs: x,s=e(x); skips.append(s)
                x=self.bot(x)
                for d,s in zip(self.decs,reversed(skips)): x=d(x,s)
                return self.head(x)

        def load_heatmap_model(path):
            ck=torch.load(path, map_location="cpu")
            m=TMJHeatmapDetector(); m.load_state_dict(ck["model_state_dict"]); m.eval().to(device)
            print(f"  {Path(path).name}: ep={ck.get('epoch','?')} MAE={ck.get('best_val_mae',float('nan')):.2f}ds")
            return m

        print("Загружаю детекторы...")
        left_det  = load_heatmap_model(str(LEFT_DETECTOR_PATH))
        right_det = load_heatmap_model(str(RIGHT_DETECTOR_PATH))

        def prep_volume(vol):
            p2,p98 = np.percentile(vol,[2,98])
            v = np.clip(vol,p2,p98); v=(v-p2)/max(p98-p2,1e-6)
            orig = np.array(v.shape,dtype=float)
            if tuple(v.shape)!=TARGET_SHAPE:
                z=[t/s for t,s in zip(TARGET_SHAPE,v.shape)]
                v=ndimage.zoom(v.astype(np.float32),z,order=1)
            return torch.tensor(v,dtype=torch.float32).unsqueeze(0).unsqueeze(0), orig

        def argmax_orig(hm, orig_shape):
            idx=np.unravel_index(hm.argmax(),hm.shape)
            c=np.array(idx,dtype=float)
            sc=orig_shape/np.array(TARGET_SHAPE,dtype=float)
            return np.clip((c*sc).astype(int),0,orig_shape.astype(int)-1)

        def extract_crop(vol, center, sz=128):
            D,H,W=vol.shape; h=sz//2
            z,y,x=int(center[0]),int(center[1]),int(center[2])
            crop=vol[max(0,z-h):min(D,z+h), max(0,y-h):min(H,y+h), max(0,x-h):min(W,x+h)]
            if crop.shape!=(sz,sz,sz):
                pad=np.zeros((sz,sz,sz),dtype=crop.dtype)
                pz=(sz-crop.shape[0])//2; py=(sz-crop.shape[1])//2; px_=(sz-crop.shape[2])//2
                pad[pz:pz+crop.shape[0],py:py+crop.shape[1],px_:px_+crop.shape[2]]=crop
                crop=pad
            return crop

        for rec in tqdm(all_records, desc="Generating crops v2"):
            sid = rec["study_id"]
            out_dir = CROPS_DIR / sid
            if (out_dir / f"{sid}_left.nii.gz").exists() and (out_dir / f"{sid}_right.nii.gz").exists():
                continue
            out_dir.mkdir(parents=True, exist_ok=True)
            raw = load_dicom_volume(rec["dicom_dir"])
            inp, orig = prep_volume(raw)
            inp = inp.to(device)
            with torch.no_grad():
                lh = torch.sigmoid(left_det(inp)).squeeze().cpu().numpy()
                rh = torch.sigmoid(right_det(inp)).squeeze().cpu().numpy()
            lc = argmax_orig(lh, orig)
            rc = argmax_orig(rh, orig)
            nib.save(nib.Nifti1Image(extract_crop(raw, lc), np.eye(4)), str(out_dir/f"{sid}_left.nii.gz"))
            nib.save(nib.Nifti1Image(extract_crop(raw, rc), np.eye(4)), str(out_dir/f"{sid}_right.nii.gz"))
        print("Готово!")
else:
    print("Кропы уже есть — пропускаем.")

In [ ]:
missing = [r for r in binary_records if not Path(r["crop_path"]).exists()]
print(f"Кропов не найдено: {len(missing)} из {len(binary_records)}")
if missing:
    print("Примеры:", [r["crop_path"] for r in missing[:3]])

## 4. Dataset & DataLoader

In [ ]:
import random as _random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
from tqdm.notebook import tqdm


def augment_volume(vol: np.ndarray) -> np.ndarray:
    """
    Консервативная аугментация для малого датасета (142 сэмпла).
    Только операции, которые не искажают анатомию.
    """
    # Random axis flips — анатомически безопасно
    for ax in range(3):
        if _random.random() < 0.5:
            vol = np.flip(vol, axis=ax).copy()

    # Лёгкий intensity jitter (контраст ±5%, яркость ±3%)
    # Симулирует вариацию протокола/аппарата сканирования
    if _random.random() < 0.7:
        alpha = _random.uniform(0.95, 1.05)
        beta  = _random.uniform(-0.03, 0.03)
        vol   = np.clip(alpha * vol + beta, 0.0, 1.0)

    # Лёгкий Gaussian noise (sigma=0.01) — регуляризация
    if _random.random() < 0.5:
        vol = np.clip(vol + np.random.normal(0, 0.01, vol.shape).astype(np.float32), 0.0, 1.0)

    return vol.astype(np.float32)


class TMJBinaryPositionDataset(Dataset):
    def __init__(self, records, is_train=False):
        self.records  = records
        self.is_train = is_train
        print(f"Кеширую {len(records)} томов в RAM...")
        self.cache = []
        for rec in tqdm(records, leave=False):
            img = nib.load(rec["crop_path"])
            vol = np.asarray(img.dataobj, dtype=np.float32)
            p2, p98 = np.percentile(vol, [2, 98])
            vol = np.clip(vol, p2, p98)
            denom = p98 - p2
            vol = (vol - p2) / denom if denom > 0 else np.zeros_like(vol)
            self.cache.append(vol)
        ram_gb = sum(v.nbytes for v in self.cache) / 1e9
        print(f"Кеш готов: {len(self.cache)} томов, {ram_gb:.2f} GB RAM")

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        vol = self.cache[idx].copy()
        if self.is_train:
            vol = augment_volume(vol)
        return (
            torch.from_numpy(vol).float().unsqueeze(0),
            torch.tensor([self.records[idx]["sag"], self.records[idx]["fr"]], dtype=torch.long)
        )


BATCH_SIZE  = 8
NUM_WORKERS = 0

train_ds = TMJBinaryPositionDataset(train_records, is_train=True)
val_ds   = TMJBinaryPositionDataset(val_records,   is_train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

print(f"Train: {len(train_ds)} сэмплов, {len(train_loader)} батчей")
print(f"Val:   {len(val_ds)} сэмплов, {len(val_loader)} батчей")

vol, lbl = next(iter(train_loader))
print(f"volume: {vol.shape} [{vol.min():.2f}, {vol.max():.2f}]  labels: {lbl[:4]}")


## 5. Model

In [ ]:
import torch.nn as nn
from typing import List, Optional

def _conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        nn.MaxPool3d(2, 2),
    )

class TMJSagittalClassifier(nn.Module):
    """Single-head binary classifier for sagittal TMJ position (central vs non-central)."""
    def __init__(self, in_channels=1, features=None, fc_hidden=256, dropout=0.5):
        super().__init__()
        if features is None: features = [16, 32, 64, 128]
        blocks, prev = [], in_channels
        for oc in features:
            blocks.append(_conv_block(prev, oc)); prev = oc
        self.backbone    = nn.Sequential(*blocks)
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Linear(prev, fc_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(fc_hidden, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        f = self.global_pool(f).view(f.size(0), -1)
        return self.head(f)  # (B, 1) logit

model = TMJSagittalClassifier().to(device)
n = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n/1e6:.2f}M")
with torch.no_grad():
    logit = model(torch.randn(1,1,32,48,48).to(device))
print(f"Output: {logit.shape}")

## 6. Training

In [ ]:
import torch.nn.functional as F

class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        if reduction not in ("mean","sum","none"): raise ValueError(f"Bad reduction: {reduction}")
        self.gamma, self.alpha, self.reduction = gamma, alpha, reduction
    def forward(self, logits, targets):
        if logits.dim()==2 and logits.shape[1]==1: logits = logits.squeeze(1)
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits); p_t = p*targets + (1-p)*(1-targets)
        loss = (1-p_t).pow(self.gamma) * bce
        if self.alpha is not None:
            loss = (self.alpha*targets + (1-self.alpha)*(1-targets)) * loss
        return loss.mean() if self.reduction=="mean" else (loss.sum() if self.reduction=="sum" else loss)

# Hyperparams
EPOCHS=150; LR=1e-4; WEIGHT_DECAY=1e-4; LR_PATIENCE=15; EARLY_STOPPING=40; GAMMA=2.0

# Class distribution
n0 = sum(1 for r in train_records if r["sag"]==0)
n1 = sum(1 for r in train_records if r["sag"]==1)
total = n0 + n1
alpha_sag = n0 / total
print(f"Train sag: 0={n0} ({100*n0/total:.1f}%)  1={n1} ({100*n1/total:.1f}%)  α={alpha_sag:.3f}")

n0v = sum(1 for r in val_records if r["sag"]==0)
n1v = sum(1 for r in val_records if r["sag"]==1)
print(f"Val   sag: 0={n0v} ({100*n0v/max(n0v+n1v,1):.1f}%)  1={n1v} ({100*n1v/max(n0v+n1v,1):.1f}%)")

In [ ]:
import datetime, json as _json2
import torch.optim as optim
from sklearn.metrics import roc_auc_score

criterion = BinaryFocalLoss(gamma=GAMMA, alpha=alpha_sag)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=LR_PATIENCE)

# Mixed precision for V100
scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
print(f"Mixed precision: {'ON (V100)' if scaler else 'OFF'}")

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
exp_dir = OUTPUT_DIR / f"sag_only_{timestamp}"
exp_dir.mkdir(parents=True, exist_ok=True)

config = {
    "task": "sagittal_only_binary",
    "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "gamma": GAMMA, "alpha_sag": alpha_sag, "batch_size": BATCH_SIZE,
    "split_ratio": SPLIT_RATIO, "train_samples": len(train_ds), "val_samples": len(val_ds),
    "train_sag_0": n0, "train_sag_1": n1, "val_sag_0": n0v, "val_sag_1": n1v,
}
with open(exp_dir/"config.json","w") as f: _json2.dump(config,f,indent=2)
print(f"Эксперимент: {exp_dir}")

In [ ]:
from tqdm.notebook import tqdm

def compute_detailed_metrics(logits, labels, thresh=0.5):
    """Accuracy, sensitivity (recall class 1), specificity (recall class 0), F1."""
    probs = torch.sigmoid(logits.squeeze(1))
    preds = (probs >= thresh).long()
    labels = labels.long()

    tp = ((preds==1) & (labels==1)).sum().item()
    tn = ((preds==0) & (labels==0)).sum().item()
    fp = ((preds==1) & (labels==0)).sum().item()
    fn = ((preds==0) & (labels==1)).sum().item()

    acc  = (tp+tn) / max(tp+tn+fp+fn, 1)
    sens = tp / max(tp+fn, 1)   # recall for non-central
    spec = tn / max(tn+fp, 1)   # recall for central
    prec = tp / max(tp+fp, 1)
    f1   = 2*prec*sens / max(prec+sens, 1e-8)
    return {"acc": acc, "sensitivity": sens, "specificity": spec, "f1": f1, "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def run_epoch(train_mode, epoch):
    model.train() if train_mode else model.eval()
    loader = train_loader if train_mode else val_loader
    tag = "Train" if train_mode else "Val  "

    rl = 0.0
    all_logits, all_labels_list = [], []

    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        for vols, labels in tqdm(loader, desc=f"[{epoch}] {tag}", leave=False):
            vols, labels = vols.to(device), labels.to(device)
            sag_labels = labels[:, 0].float()

            if train_mode:
                optimizer.zero_grad()
                if scaler:
                    with torch.cuda.amp.autocast():
                        logit = model(vols)
                        loss = criterion(logit, sag_labels)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer); scaler.update()
                else:
                    logit = model(vols)
                    loss = criterion(logit, sag_labels)
                    loss.backward(); optimizer.step()
            else:
                logit = model(vols)
                loss = criterion(logit, sag_labels)

            rl += loss.item()
            all_logits.append(logit.detach().float().cpu())
            all_labels_list.append(labels[:, 0].cpu())

    all_logits = torch.cat(all_logits)
    all_labels_t = torch.cat(all_labels_list)

    m = compute_detailed_metrics(all_logits, all_labels_t)
    m["loss"] = rl / len(loader)

    # AUC (only if both classes present)
    probs_np = torch.sigmoid(all_logits.squeeze(1)).numpy()
    labels_np = all_labels_t.numpy()
    if len(set(labels_np.tolist())) > 1:
        m["auc"] = float(roc_auc_score(labels_np, probs_np))
    else:
        m["auc"] = float("nan")

    return m

best_val_acc = -1.0; no_imp = 0; history = []; best_path = exp_dir / "best_model.pth"

print(f"\n{'Ep':>4} {'Tr-loss':>8} {'Tr-acc':>7} {'Tr-sens':>8} {'Tr-spec':>8} │ "
      f"{'Va-loss':>8} {'Va-acc':>7} {'Va-sens':>8} {'Va-spec':>8} {'Va-F1':>7} {'Va-AUC':>7}")
print("─" * 95)

for epoch in range(1, EPOCHS+1):
    tr  = run_epoch(True,  epoch)
    val = run_epoch(False, epoch)
    scheduler.step(val["acc"]); lr_now = optimizer.param_groups[0]["lr"]

    print(f"{epoch:>4} {tr['loss']:>8.4f} {tr['acc']:>7.3f} {tr['sensitivity']:>8.3f} {tr['specificity']:>8.3f} │ "
          f"{val['loss']:>8.4f} {val['acc']:>7.3f} {val['sensitivity']:>8.3f} {val['specificity']:>8.3f} "
          f"{val['f1']:>7.3f} {val['auc']:>7.3f}  lr={lr_now:.1e}")

    row = {"epoch": epoch, "lr": lr_now}
    row.update({f"train_{k}": v for k,v in tr.items()})
    row.update({f"val_{k}": v for k,v in val.items()})
    history.append(row)
    with open(exp_dir/"metrics.jsonl","a") as f: f.write(_json2.dumps(row)+"\n")

    if val["acc"] > best_val_acc:
        best_val_acc = val["acc"]; no_imp = 0
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "best_val_accuracy": best_val_acc, "val_metrics": val}, best_path)
        print(f"     ✓ best (acc={best_val_acc:.3f} sens={val['sensitivity']:.3f} spec={val['specificity']:.3f} AUC={val['auc']:.3f})")
    else:
        no_imp += 1
        if EARLY_STOPPING > 0 and no_imp >= EARLY_STOPPING:
            print(f"     Early stop at epoch {epoch}"); break

print(f"\nГотово. Best val acc: {best_val_acc:.3f}")

## 7. Threshold Calibration

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report

ckpt = torch.load(best_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
print(f"Best epoch: {ckpt['epoch']}, val acc: {ckpt['best_val_accuracy']:.3f}")
print(f"Val metrics at best: {ckpt['val_metrics']}")

all_probs, all_labels_cal = [], []
with torch.no_grad():
    for vols, labels in val_loader:
        logit = model(vols.to(device))
        all_probs.extend(torch.sigmoid(logit.squeeze(1)).cpu().tolist())
        all_labels_cal.extend(labels[:, 0].tolist())

probs_np  = np.array(all_probs)
labels_np = np.array(all_labels_cal)

print(f"\nVal distribution: 0(central)={sum(labels_np==0)}, 1(non-central)={sum(labels_np==1)}")
print(f"Predictions >0.5: {sum(probs_np>0.5)}")

fpr, tpr, thresholds = roc_curve(labels_np, probs_np)
auc = roc_auc_score(labels_np, probs_np)
j_scores = tpr - fpr
best_idx  = int(np.argmax(j_scores))
best_thresh = float(thresholds[best_idx])
acc_at_thresh = float(np.mean((probs_np >= best_thresh) == labels_np))

print(f"\nAUC-ROC: {auc:.4f}")
print(f"Youden's J best threshold: {best_thresh:.4f}")
print(f"Accuracy at threshold: {acc_at_thresh:.4f}")

preds_opt = (probs_np >= best_thresh).astype(int)
print(f"\nConfusion matrix (threshold={best_thresh:.3f}):")
print(confusion_matrix(labels_np, preds_opt, labels=[0,1]))
print(classification_report(labels_np, preds_opt, labels=[0,1], target_names=["central","non-central"]))

calibration = {"optimal_threshold": best_thresh, "auc_roc": auc, "accuracy_at_threshold": acc_at_thresh}
with open(exp_dir/"config.json") as f: cfg = _json2.load(f)
cfg.update(calibration); cfg["best_val_accuracy"] = best_val_acc
with open(exp_dir/"config.json","w") as f: _json2.dump(cfg,f,indent=2)
print(f"Порог сохранён: {best_thresh:.4f}")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt

ep = [h["epoch"] for h in history]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Sagittal-Only Binary Classifier", fontsize=14, fontweight="bold")

axes[0,0].plot(ep, [h["train_loss"] for h in history], label="Train")
axes[0,0].plot(ep, [h["val_loss"]   for h in history], label="Val")
axes[0,0].set_title("Loss"); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep, [h["train_acc"] for h in history], label="Train")
axes[0,1].plot(ep, [h["val_acc"]   for h in history], label="Val")
axes[0,1].axhline(0.775, color="red", linestyle="--", alpha=0.5, label="Prev best (0.775)")
axes[0,1].set_title("Accuracy"); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(ep, [h["val_auc"] for h in history if not np.isnan(h.get("val_auc", float("nan")))],
               label="Val AUC")
axes[0,2].axhline(0.726, color="red", linestyle="--", alpha=0.5, label="Prev AUC (0.726)")
axes[0,2].set_title("AUC-ROC"); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

axes[1,0].plot(ep, [h["train_sensitivity"] for h in history], label="Train Sensitivity")
axes[1,0].plot(ep, [h["val_sensitivity"]   for h in history], label="Val Sensitivity")
axes[1,0].set_title("Sensitivity (non-central recall)"); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ep, [h["train_specificity"] for h in history], label="Train Specificity")
axes[1,1].plot(ep, [h["val_specificity"]   for h in history], label="Val Specificity")
axes[1,1].set_title("Specificity (central recall)"); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

axes[1,2].plot(ep, [h["val_f1"] for h in history], label="Val F1", color="purple")
axes[1,2].set_title("Val F1"); axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(exp_dir/"training_report.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {exp_dir}/training_report.png")

# ROC curve
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, lw=2, label=f"AUC={auc:.3f}")
ax.plot([0,1],[0,1],"k--",alpha=0.4)
ax.scatter([fpr[best_idx]], [tpr[best_idx]], color="red", s=100, zorder=5, label=f"thresh={best_thresh:.3f}")
ax.set_xlabel("FPR (1-Specificity)"); ax.set_ylabel("TPR (Sensitivity)")
ax.set_title("ROC — Sagittal (central vs non-central)")
ax.legend(); ax.grid(alpha=0.3)
fig.savefig(exp_dir/"roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Export

In [ ]:
analysis = {
    "config": cfg,
    "total_epochs": len(history),
    "best_epoch": ckpt["epoch"],
    "best_val_accuracy": best_val_acc,
    "calibration": calibration,
    "history": history,
}
ap = exp_dir / "training_analysis.json"
with open(ap, "w") as f: _json2.dump(analysis, f, indent=2, ensure_ascii=False)
print(f"Analysis: {ap}")
print(f"Model:    {best_path}")
print(f"\nИтог — Best val acc: {best_val_acc:.3f} (baseline: 0.680)")
for n in HEAD_NAMES:
    print(f"  [{n}] AUC={calibration['auc_roc'][n]}  thresh={calibration['optimal_thresholds'][n]}")